# Flight Delay Intelligence System — Data Understanding
This notebook documents the initial inspection of the BTS On-Time Performance dataset (2014–2018, 3M sampled flights). Goal is to understand structure, column meanings, and data quality before EDA.

In [1]:
import pandas as pd
import numpy as np

## 1. Load Data
Loading the working dataset sampled from 30M flights across 5 years (2014–2018).

In [ ]:
df=pd.read_csv(r"data/raw/flights_working.csv")

## 2. Dataset Overview
Checking shape, column names, and data types to confirm structure.

In [20]:
df.head()

,FL_DATE,OP_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,DEST,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,TAXI_OUT,WHEELS_OFF,...,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,Unnamed: 27
0,2014-06-14,F9,407,DEN,LAX,1720,1826.0,66.0,10.0,1836.0,...,140.0,140.0,120.0,862.0,0.0,0.0,32.0,0.0,34.0,NaN
1,2014-07-21,WN,4704,CMH,BWI,1555,1555.0,0.0,16.0,1611.0,...,75.0,76.0,55.0,337.0,NaN,NaN,NaN,NaN,NaN,NaN
2,2014-10-09,AA,43,DTW,DFW,1450,1456.0,6.0,11.0,1507.0,...,175.0,171.0,141.0,986.0,NaN,NaN,NaN,NaN,NaN,NaN
3,2014-10-27,B6,1076,MCO,PVD,1245,1239.0,-6.0,11.0,1250.0,...,159.0,156.0,141.0,1072.0,NaN,NaN,NaN,NaN,NaN,NaN
4,2014-04-22,WN,330,LAS,CLE,1525,1545.0,20.0,9.0,1554.0,...,245.0,224.0,211.0,1824.0,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Descriptive Statistics
Key observations:
- `ARR_DELAY` mean is ~9.7 min but median is -2 min — most flights are on time or early, distribution is right-skewed
- `DEP_DELAY` min is -136 min, max is 2482 min — extreme outliers exist on both ends
- `CRS_ELAPSED_TIME` has a minimum of -60 — negative flight duration is impossible, needs handling in preprocessing
- Time columns (`CRS_DEP_TIME`, `DEP_TIME` etc.) are stored as HHMM integers, not actual time objects

In [4]:
df.shape

(3000000, 28)

In [5]:
df.columns

Index(['FL_DATE', 'OP_CARRIER', 'OP_CARRIER_FL_NUM', 'ORIGIN', 'DEST',
       'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'TAXI_OUT', 'WHEELS_OFF',
       'WHEELS_ON', 'TAXI_IN', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY',
       'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'CRS_ELAPSED_TIME',
       'ACTUAL_ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'CARRIER_DELAY',
       'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY',
       'Unnamed: 27'],
      dtype='str')

In [8]:
df.shape

(3000000, 27)

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000000 entries, 0 to 2999999
Data columns (total 27 columns):
 #   Column               Dtype  
---  ------               -----  
 0   FL_DATE              str    
 1   OP_CARRIER           str    
 2   OP_CARRIER_FL_NUM    int64  
 3   ORIGIN               str    
 4   DEST                 str    
 5   CRS_DEP_TIME         int64  
 6   DEP_TIME             float64
 7   DEP_DELAY            float64
 8   TAXI_OUT             float64
 9   WHEELS_OFF           float64
 10  WHEELS_ON            float64
 11  TAXI_IN              float64
 12  CRS_ARR_TIME         int64  
 13  ARR_TIME             float64
 14  ARR_DELAY            float64
 15  CANCELLED            float64
 16  CANCELLATION_CODE    str    
 17  DIVERTED             float64
 18  CRS_ELAPSED_TIME     float64
 19  ACTUAL_ELAPSED_TIME  float64
 20  AIR_TIME             float64
 21  DISTANCE             float64
 22  CARRIER_DELAY        float64
 23  WEATHER_DELAY        float64
 24  NAS_DELAY

In [10]:
df.describe()

,OP_CARRIER_FL_NUM,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,...,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
count,3.000000e+06,3.000000e+06,2.953749e+06,2.953343e+06,2.952417e+06,2.952417e+06,2.950860e+06,2.950860e+06,3.000000e+06,2.950860e+06,...,3.000000e+06,2.999996e+06,2.944653e+06,2.944653e+06,3.000000e+06,562392.000000,562392.000000,562392.000000,562392.000000,562392.000000
mean,2.261915e+03,1.329235e+03,1.334285e+03,9.737405e+00,1.642995e+01,1.356759e+03,1.469037e+03,7.430115e+00,1.492072e+03,1.473934e+03,...,2.466333e-03,1.424149e+02,1.379040e+02,1.140556e+02,8.253643e+02,19.029001,2.852032,14.641569,0.080083,24.503202
std,1.791470e+03,4.849757e+02,4.983551e+02,4.070280e+01,9.188262e+00,4.998939e+02,5.255512e+02,5.735615e+00,5.098029e+02,5.296651e+02,...,4.960092e-02,7.534145e+01,7.476595e+01,7.264735e+01,6.108821e+02,54.162256,23.312247,31.645043,2.289161,45.651352
min,1.000000e+00,1.000000e+00,1.000000e+00,-1.360000e+02,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,...,0.000000e+00,-6.000000e+01,1.400000e+01,8.000000e+00,3.000000e+01,0.000000,0.000000,0.000000,0.000000,0.000000
25%,7.650000e+02,9.150000e+02,9.200000e+02,-5.000000e+00,1.100000e+01,9.340000e+02,1.052000e+03,4.000000e+00,1.108000e+03,1.056000e+03,...,0.000000e+00,8.700000e+01,8.300000e+01,6.100000e+01,3.720000e+02,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1.770000e+03,1.323000e+03,1.328000e+03,-2.000000e+00,1.400000e+01,1.341000e+03,1.507000e+03,6.000000e+00,1.520000e+03,1.511000e+03,...,0.000000e+00,1.230000e+02,1.190000e+02,9.500000e+01,6.500000e+02,1.000000,0.000000,2.000000,0.000000,4.000000
75%,3.466000e+03,1.730000e+03,1.740000e+03,7.000000e+00,1.900000e+01,1.755000e+03,1.912000e+03,9.000000e+00,1.918000e+03,1.917000e+03,...,0.000000e+00,1.740000e+02,1.700000e+02,1.440000e+02,1.065000e+03,18.000000,0.000000,18.000000,0.000000,30.000000
max,9.793000e+03,2.359000e+03,2.400000e+03,2.482000e+03,2.170000e+02,2.400000e+03,2.400000e+03,4.110000e+02,2.400000e+03,2.400000e+03,...,1.000000e+00,7.180000e+02,7.640000e+02,7.040000e+02,4.983000e+03,1810.000000,2475.000000,1515.000000,437.000000,1445.000000


## 4. Categorical Column Analysis
Inspecting unique carriers, airports, and cancellation codes.
- 20 unique carriers — WN (Southwest) dominates with ~21% of flights
- 367 unique origin and destination airports — ATL is the busiest hub
- Cancellation codes: B (Weather) is most common, D (Security) is extremely rare with only 39 cases

In [13]:
df['OP_CARRIER'].value_counts()

OP_CARRIER
WN    642914
DL    447439
AA    398978
OO    328229
UA    275209
EV    235369
B6    139946
MQ     95973
AS     92779
US     63661
NK     57927
F9     49423
HA     39135
VX     28878
YX     26467
OH     23129
9E     20431
YV     17727
FL      8324
G4      8062
Name: count, dtype: int64

In [15]:
df['ORIGIN'].value_counts()

ORIGIN
ATL    189186
ORD    143583
DFW    119169
DEN    112429
LAX    108672
        ...  
HGR        11
DRT        10
OWB        10
CYS         3
ENV         1
Name: count, Length: 367, dtype: int64

In [16]:
print(df['DEST'].nunique())
print(df['CANCELLATION_CODE'].value_counts(dropna=False))

367
CANCELLATION_CODE
NaN    2952051
B        25289
A        13087
C         9534
D           39
Name: count, dtype: int64


## 5. Date Range
Confirming temporal coverage of the dataset.
- Range: 2014-01-01 to 2018-12-31 — exactly 5 years
- 1826 unique dates — accounts for all calendar days including 2016 leap year, no gaps

In [17]:
print(df['FL_DATE'].min())
print(df['FL_DATE'].max())
print(df['FL_DATE'].nunique())

2014-01-01
2018-12-31
1826


## 6. Missing Values
- `Unnamed: 27` — 100% null, artifact column, will be dropped in preprocessing
- `CANCELLATION_CODE` — 98.4% null by design, only populated for cancelled flights (~1.6% of data)
- Delay cause columns (`CARRIER_DELAY`, `WEATHER_DELAY`, `NAS_DELAY`, `SECURITY_DELAY`, `LATE_AIRCRAFT_DELAY`) — 81.25% null by design, only populated when a delay occurred
- `ARR_DELAY`, `ARR_TIME`, `DEP_DELAY`, `DEP_TIME` — ~1.5–1.85% null, corresponds to cancelled/diverted flights that never arrived
- `CRS_ELAPSED_TIME` — 4 rows null, negligible

In [22]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
print(pd.DataFrame({'count': missing, 'pct': missing_pct})
      .query('count > 0')
      .sort_values('pct', ascending=False))

                       count     pct
Unnamed: 27          3000000  100.00
CANCELLATION_CODE    2952051   98.40
CARRIER_DELAY        2437608   81.25
SECURITY_DELAY       2437608   81.25
LATE_AIRCRAFT_DELAY  2437608   81.25
NAS_DELAY            2437608   81.25
WEATHER_DELAY        2437608   81.25
ARR_DELAY              55539    1.85
ACTUAL_ELAPSED_TIME    55347    1.84
AIR_TIME               55347    1.84
ARR_TIME               49140    1.64
TAXI_IN                49140    1.64
WHEELS_ON              49140    1.64
WHEELS_OFF             47583    1.59
TAXI_OUT               47583    1.59
DEP_DELAY              46657    1.56
DEP_TIME               46251    1.54
CRS_ELAPSED_TIME           4    0.00


## 7. Key Findings

**Confirmed:**
- Dataset is flight-level (one row = one flight) ✅
- 3,000,000 flights across 20 airlines, 367 airports, 5 years ✅
- All missing values are logically explained — no unexpected nulls ✅

**Issues to fix in preprocessing:**
- Drop `Unnamed: 27` (100% null)
- Parse `FL_DATE` to datetime, extract `YEAR`, `MONTH`, `DAY`, `DAY_OF_WEEK`
- Convert `CRS_DEP_TIME` / `CRS_ARR_TIME` from HHMM integer to hour
- Handle negative `CRS_ELAPSED_TIME` (4 rows + impossible values)
- Fill delay cause nulls with 0 for non-delayed flights
- Exclude cancelled flights from delay prediction model
- Create target variable: `ARR_DELAY >= 15` → binary 0/1